# Fine-tune a logo detector on Colab

Trains a YOLO model with `sports_analyzer.train` on a dataset version built by `scripts/prepare_dataset.py`. At the end, Ultralytics validates the best epoch and prints precision, recall and mAP per brand.

**Disconnects are fine.** Training output is saved to Google Drive after every epoch. If the session drops or you stop it, reconnect and run all cells again: training resumes after the last completed epoch, and a finished run is left alone.

Runs are named after their dataset, model and image size (e.g. `f1_v01_yolo11s_1280`), so a new dataset version or different settings start a new run.

Before running, upload the dataset zip (e.g. `data/datasets/f1_v01.zip`) to `MyDrive/sports-analyzer/datasets/`. Use a GPU runtime: Runtime → Change runtime type → T4 GPU.

In [ ]:
# Settings
DRIVE_DIR = "/content/drive/MyDrive/sports-analyzer"  # everything that must survive a disconnect lives here
BRANCH = "main"  # branch of the GitHub repo to install the code from
DATASET = "f1_v01"  # DRIVE_DIR/datasets/f1_v01.zip
MODEL = "yolo11s.pt"
EPOCHS = 150
IMGSZ = 1280

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
# Same Ultralytics version as uv.lock, so the trained weights load locally
%pip install -q "sports-analyzer @ git+https://github.com/AndrewKM210/sports-analyser@{BRANCH}" ultralytics==8.4.174

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Copy the dataset from Drive to the Colab disk, which is much faster to read during training.

In [ ]:
import zipfile
from pathlib import Path

dataset_dir = Path("/content/datasets") / DATASET
zipfile.ZipFile(f"{DRIVE_DIR}/datasets/{DATASET}.zip").extractall(dataset_dir)
data_yaml = next(dataset_dir.rglob("data.yaml"))  # works whatever the folder inside the zip is called

## Train

Starts the run, resumes it after a disconnect, or does nothing if it already finished. Weights and plots go to `MyDrive/sports-analyzer/runs/train/<run name>/`.

In [ ]:
from sports_analyzer.train import run_name, train

name = run_name(f"{DATASET}/data.yaml", MODEL, IMGSZ)  # named after the dataset version, not the zip's folder
run_dir = train(
    str(data_yaml),
    model=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=-1,  # pick the largest batch that fits in GPU memory
    project=f"{DRIVE_DIR}/runs/train",
    name=name,
)

## Use the model locally

Download the run folder from `MyDrive/sports-analyzer/runs/train/` into `runs/train/` locally, then:

```bash
uv run sports-analyzer evaluate data/datasets/f1_v01/data.yaml --sport f1 --detector finetuned \
  --weights runs/train/f1_v01_yolo11s_1280/weights/best.pt
uv run sports-analyzer analyze data/videos/f1.mp4 --sport f1 --detector finetuned \
  --weights runs/train/f1_v01_yolo11s_1280/weights/best.pt
```